# AEGIS: activity repair with an LTLf semantic loss and hard constraints

Same pipeline as `main.py`, cell by cell:

1. load the repair graphs of one dataset;
2. mine DECLARE/LTLf rules on the complete training traces and compile them to DFAs;
3. build the **semantic loss** (all mined rules) and the **hard-constrained decoder** (rules with train support >= `SUP_MIN`);
4. pick hyperparameters (shipped config or Optuna search);
5. train `NUM_RUNS` networks with the semantic loss, or evaluate the pretrained ones (`EVAL_ONLY = True`),
   and report SL (argmax) and SL + hard constraints on the five test sets.

With the defaults below the notebook only evaluates the pretrained models (a few minutes on a GPU).

In [ ]:
import json

import numpy as np
import pandas as pd
import torch

from aegis import data as D
from aegis import rules as R
from aegis import ltlf_features as LF
from aegis.experiment import MODEL_DIR, evaluate_saved, test_multi

pd.set_option("display.max_columns", None)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

## Settings

In [ ]:
DATASET = "bpi_2013_CZ"   # one of D.DATASETS
TAG = "sl"                # trained_model/<dataset>_AT_only/<TAG>/ and results/repair/<dataset>_AT_only/*_<TAG>.csv
EVAL_ONLY = True          # True: evaluate the pretrained models of TAG; False: train new ones (use a new TAG)
RUN_HPO = False           # True: Optuna search (30 trials x 200 epochs) instead of the shipped config
NUM_RUNS, EPOCHS = 10, 200
LAMBDA_SEM = 0.3          # weight of the semantic loss
SUP_MIN = 0.99            # hard constraints: rules with train support >= SUP_MIN
BEAM = 16                 # beam width of the hard decoder
SEED = 0                  # run i uses SEED + i
print(D.DATASETS)

## Data

In [ ]:
D.seed_everything(SEED)
data = D.load_dataset(DATASET)
tests = D.load_test_sets(DATASET)
print(data.outputcat, data.outputreal, "| edge types:", data.edge_types)
print({tt: len(X) for tt, X in tests.items()})

## Rules, semantic loss and hard decoder

Rules are mined on the complete training traces (labels) in a compact alphabet `0..A-1` of the real activities.
The semantic loss `-log P(rule accepts)` runs each DFA on the per-position distributions of the network
(Xu et al. 2018) and only counts (trace, rule) pairs the ground truth satisfies. The hard decoder searches jointly over
all missing positions of a trace (beam search, one DFA state per rule) and keeps only completions every enforced rule
can still accept; rules the visible events already violate are ignored for that trace.

In [ ]:
rules = R.mine(data, top_k=60)
sem_loss = R.semantic_loss(data, rules, device)
decoder, ridx = R.hard_decoder(data, rules, SUP_MIN)
print(f"{len(rules.dfas)} rules in the semantic loss, {len(ridx)} enforced by the hard decoder (support >= {SUP_MIN})")

### Optional: how much do the rules restrict the missing activities?
Per missing node of the test set: activities left by the enforced rules (each rule checked on its own; a rule the visible events
already violate is ignored), and how often the true activity would be excluded.

In [ ]:
def diagnose(graphs, dfas, n=3000):
    A = data.a_real
    n_miss = n_excl_any = n_excl_true = 0
    allowed = []
    for g in graphs[:n]:
        acts = data.act_ids(g)
        m = [i for i, a in enumerate(acts) if a == A]
        if not m:
            continue
        f = LF.rule_features(dfas, acts, A).reshape(len(acts), len(dfas), A + 1)
        mask = ((f[:, :, :A] > 0) | (f[:, :, A:] > 0)).all(1)    # unsatisfiable rule -> no restriction
        y = data.compact(g.y[D.ACT_KEY])
        for i in m:
            n_miss += 1
            allowed.append(int(mask[i].sum()))
            n_excl_any += mask[i].sum() < A
            n_excl_true += not mask[i, y[i]]
    print(f"missing nodes: {n_miss} | mean #activities left (of {A}): {np.mean(allowed):.2f}")
    print(f"rules exclude at least one activity at {100 * n_excl_any / n_miss:.1f}% of missing nodes")
    print(f"rules exclude the TRUE activity at   {100 * n_excl_true / n_miss:.2f}% of missing nodes")


diagnose(data.X_test, [rules.dfas[k] for k in ridx])

## Hyperparameters

In [ ]:
if EVAL_ONLY:
    config = None
elif RUN_HPO:
    from aegis.hpo import run_hpo
    study, config = run_hpo(data, device, sem_loss, lambda_sem=LAMBDA_SEM, n_trials=30, epochs=200, seed=SEED)
else:
    config = D.load_config(DATASET)
if config is not None:
    config = {**config, "lambda_sem": LAMBDA_SEM}
config

## Train and evaluate

`acc_sl`: argmax of the network trained with the semantic loss. `acc_sl_hard`: hard-constrained decoding of the same
output. `sat_*`: share of (trace, rule) pairs satisfied by the completed trace (rules true on the ground truth only).

In [ ]:
if EVAL_ONLY:
    res, summary = evaluate_saved(data, tests, decoder, device, tag=TAG, beam=BEAM)
else:
    assert not any((MODEL_DIR / data.name / TAG).glob("run*.pt")), f"{TAG} already has trained models: use a new TAG"
    res, summary = test_multi(config, data, tests, sem_loss, decoder, device, tag=TAG, num_runs=NUM_RUNS,
                              num_epochs=EPOCHS, beam=BEAM, sup_min=SUP_MIN, seed=SEED)
summary

## Comparison with the published numbers (`results_reference/`)

In [ ]:
ref = pd.read_csv(D.ROOT / "results_reference" / "repair" / data.name / f"{data.name}_summary_sl.csv")
cols = ["acc_sl", "acc_sl_hard", "timestamp_mae"]
pd.concat({"this run": summary.set_index("test_type")[cols], "published": ref.set_index("test_type")[cols]}, axis=1).round(4)

Figures: `python plot_sl_hard.py --results-dir results/repair --tag <TAG>` (accuracy and MAE against the baseline in `BASELINE.json`).